## 11.2 GRU - 前向传播&矩阵计算

#### 1. 为什么这一节要学习前向传播

##### 1.1 学完结构之后，下一步就要看“它到底怎么计算”
上一节我们已经理解了 GRU 的核心结构，知道了它有：

- 更新门 $z_t$
- 重置门 $r_t$
- 候选隐藏状态 $\tilde{h}_t$
- 当前隐藏状态 $h_t$

但这还只是“组件介绍”。

真正要把 GRU 学懂，必须继续回答下面几个问题：

- 在一个时间步里，这几个量谁先算、谁后算？
- 每一步为什么要这么算？
- 每个张量的维度是多少？
- 整个序列是如何由单步重复得到的？
- 加入 batch 之后，shape 怎么变化？

##### 1.2 GRU 的前向传播其实比 LSTM 更紧凑
如果你已经学过 LSTM，你会发现 GRU 的前向传播会更简洁，因为：

- 没有单独的细胞状态 $C_t$
- 没有三个门那么复杂
- 最终只维护一个隐藏状态 $h_t$

所以从计算流程上说：

>GRU 可以理解为“更短、更紧凑的门控前向传播”。

#### 2. 先从单时间步开始：输入和输出是什么

##### 2.1 单时间步输入
在时间步 $t$，GRU 接收两个输入：

- 当前输入：$x_t$
- 上一时刻隐藏状态：$h_{t-1}$

写成：

$(x_t,\ h_{t-1})$

这点和 LSTM 的区别很明显：

- LSTM：输入 $x_t$、$h_{t-1}$、$C_{t-1}$
- GRU：只要 $x_t$、$h_{t-1}$

##### 2.2 单时间步输出
GRU 在当前时刻只输出一个状态：

- 当前隐藏状态：$h_t$

写成：

$(x_t,\ h_{t-1}) \rightarrow h_t$

同时，这个 $h_t$ 还会继续传给下一时间步。

所以 GRU 的信息链路是：

$h_{t-1} \rightarrow h_t \rightarrow h_{t+1}$

#### 3. GRU 单时间步前向传播的完整流程

##### 3.1 总体顺序先记住
在时间步 $t$，GRU 的前向传播顺序通常是：

1. 计算更新门 $z_t$
2. 计算重置门 $r_t$
3. 计算候选隐藏状态 $\tilde{h}_t$
4. 融合得到当前隐藏状态 $h_t$

这个顺序非常重要，后面所有理解都围绕它展开。📌

##### 3.2 第一步：计算更新门 $z_t$

**（1）公式**

$z_t = \sigma(W_z x_t + U_z h_{t-1} + b_z)$

其中：

- $W_z$：输入到更新门的权重
- $U_z$：隐藏状态到更新门的权重
- $b_z$：偏置
- $\sigma$：Sigmoid 函数

**（2）更新门到底在控制什么**
更新门控制的是：

当前时刻最终状态 $h_t$，应该保留多少旧状态 $h_{t-1}$，又应该接收多少新候选状态。

它本质上是一个比例控制器。

如果某一维上：

- $z_t \rightarrow 1$：更偏向保留旧状态
- $z_t \rightarrow 0$：更偏向接收新候选状态

所以更新门是 GRU 中最核心的门。

##### 3.3 第二步：计算重置门 $r_t$

**（1）公式**

$r_t = \sigma(W_r x_t + U_r h_{t-1} + b_r)$

**（2）重置门到底在控制什么**
重置门控制的是：

在生成新的候选隐藏状态时，上一时刻隐藏状态 $h_{t-1}$ 应该参与多少。

它不是直接控制最终输出，而是影响“新内容怎么生成”。

直观理解：

- $r_t \rightarrow 0$：过去影响弱，像“先把过去忘掉一些”
- $r_t \rightarrow 1$：过去完整参与候选状态生成

所以它更像一个：

过去信息参与候选生成的开关。

##### 3.4 第三步：计算候选隐藏状态 $\tilde{h}_t$

**（1）公式**

$\tilde{h}_t = \tanh(W_h x_t + U_h(r_t \odot h_{t-1}) + b_h)$

**（2）这一项的作用是什么**
它表示：

如果当前时刻要写入新信息，那么这个“新信息内容”是什么。

也就是说：

- 更新门决定“要不要更新”
- 候选隐藏状态决定“更新成什么内容”

**（3）为什么这里要先乘 $r_t$**
这里最关键的细节是：

$r_t \odot h_{t-1}$

意思是：

上一时刻隐藏状态不会直接参与候选状态计算，而是先经过重置门筛选。

这表示：

GRU 可以先决定“参考多少过去”，然后再生成当前时刻的新候选内容。

##### 3.5 第四步：融合得到当前隐藏状态 $h_t$

**（1）公式**

$h_t = z_t \odot h_{t-1} + (1-z_t)\odot \tilde{h}_t$

**（2）这个公式是 GRU 的核心**
它表示当前隐藏状态由两部分构成：

- 旧状态的保留部分：$z_t \odot h_{t-1}$
- 新候选状态的加入部分：$(1-z_t)\odot \tilde{h}_t$

所以 GRU 并不是：

- 全部沿用旧状态
- 或者全部替换成新状态

而是：

让旧状态和新状态做加权融合。

**（3）这个融合思想为什么很重要**
因为这样做可以让模型自己学习：

- 某些信息需要长时间保留
- 某些信息应该快速更新
- 某些维度偏向“记忆”
- 某些维度偏向“变化”

这也是 GRU 能处理长期依赖的关键原因之一。

#### 4. 单时间步的维度计算

##### 4.1 先约定维度符号
设：

- 输入维度：$d_x$
- 隐藏维度：$d_h$

那么：

- $x_t \in \mathbb{R}^{d_x}$
- $h_{t-1} \in \mathbb{R}^{d_h}$

##### 4.2 更新门的维度
公式：

$z_t = \sigma(W_z x_t + U_z h_{t-1} + b_z)$

要让最后结果能够和 $h_{t-1}$ 做逐元素运算，所以：

$z_t \in \mathbb{R}^{d_h}$

因此：

- $W_z \in \mathbb{R}^{d_h \times d_x}$
- $U_z \in \mathbb{R}^{d_h \times d_h}$

##### 4.3 重置门的维度
同理：

$r_t \in \mathbb{R}^{d_h}$

所以：

- $W_r \in \mathbb{R}^{d_h \times d_x}$
- $U_r \in \mathbb{R}^{d_h \times d_h}$

##### 4.4 候选隐藏状态的维度
因为它最终要和 $h_{t-1}$ 融合，所以：

$\tilde{h}_t \in \mathbb{R}^{d_h}$

所以：

- $W_h \in \mathbb{R}^{d_h \times d_x}$
- $U_h \in \mathbb{R}^{d_h \times d_h}$

##### 4.5 当前隐藏状态维度
最终：

$h_t \in \mathbb{R}^{d_h}$

GRU 中所有门、候选状态、隐藏状态，它们的核心维度都统一为隐藏维度 $d_h$。

#### 5. 从单时间步推广到多时间步

##### 5.1 多时间步的本质
设一个序列为：

$x_1,\ x_2,\ x_3,\ \dots,\ x_T$

那么 GRU 的整个前向传播，其实就是不断重复单时间步计算：

- 第一步：$(x_1,\ h_0) \rightarrow h_1$
- 第二步：$(x_2,\ h_1) \rightarrow h_2$
- 第三步：$(x_3,\ h_2) \rightarrow h_3$
- ......
- 第 $T$ 步：$(x_T,\ h_{T-1}) \rightarrow h_T$

也就是说：

同一个 GRU 单元沿时间维不断重复调用。

##### 5.2 多时间步并没有新公式
这一点和 RNN、LSTM 一样非常重要：

整个序列并没有新的前向传播公式，本质上只是“单时间步公式的重复调用”。

#### 6. 多时间步下的维度计算

##### 6.1 单条样本的整个序列输入
如果只看一条序列，长度为 $T$，每个时间步输入维度为 $d_x$，那么：

$X \in \mathbb{R}^{T \times d_x}$

##### 6.2 整个序列输出
GRU 在每个时间步都会输出一个隐藏状态：

$h_1,\ h_2,\ \dots,\ h_T$

所以整个隐藏状态序列为：

$H \in \mathbb{R}^{T \times d_h}$

##### 6.3 最终状态
除了整个隐藏状态序列外，通常还会特别关心最后一个隐藏状态：

$h_T \in \mathbb{R}^{d_h}$

它在很多任务里都很重要，例如：

- 文本分类
- 序列编码
- Seq2Seq 编码器最终状态

#### 7. 加入 batch 之后如何理解

##### 7.1 为什么要加 batch
实际训练时不会一条序列一条序列地单独算，而是会一次输入多条样本组成一个 batch，这样可以：

- 提高 GPU 利用率
- 加速训练
- 让梯度更新更稳定

##### 7.2 batch 后输入张量维度
设：

- batch size = $B$
- 序列长度 = $T$
- 输入维度 = $d_x$

如果采用 `batch_first=True`，那么：

$X \in \mathbb{R}^{B \times T \times d_x}$

含义：

- 第 1 维：batch 中有几条样本
- 第 2 维：每条样本有几个时间步
- 第 3 维：每个时间步的特征维度

##### 7.3 固定某个时间步来看
当固定某个时间步 $t$ 时，这一时刻的输入其实是：

$x_t \in \mathbb{R}^{B \times d_x}$

同时上一时刻隐藏状态为：

$h_{t-1} \in \mathbb{R}^{B \times d_h}$

所以单时间步的逻辑没有变，只是从“一个向量”变成了“一批向量并行”。

#### 8. 加入 batch 后的维度变化

##### 8.1 各个门的维度
在 batch 情况下：

- $z_t \in \mathbb{R}^{B \times d_h}$
- $r_t \in \mathbb{R}^{B \times d_h}$
- $\tilde{h}_t \in \mathbb{R}^{B \times d_h}$
- $h_t \in \mathbb{R}^{B \times d_h}$

##### 8.2 整个序列输出维度
如果使用 `batch_first=True`，那么：

$H \in \mathbb{R}^{B \times T \times d_h}$

最后状态：

$h_T \in \mathbb{R}^{B \times d_h}$